In [ ]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

In [20]:
data =  pd.read_csv("dataset/train.txt", sep=";")
data.columns = ["Text", "Emotion"]
print(data.head())

                                                Text  Emotion
0  i can go from feeling so hopeless to so damned...  sadness
1   im grabbing a minute to post i feel greedy wrong    anger
2  i am ever feeling nostalgic about the fireplac...     love
3                               i am feeling grouchy    anger
4  ive been feeling a little burdened lately wasn...  sadness


In [21]:
texts = data["Text"].tolist()
lables = data["Emotion"].tolist()

#tokenizer the text data
# Create a Tokenizer instance
tokenizer = Tokenizer()
# Fit the tokenizer on text data
tokenizer.fit_on_texts(texts)

In [22]:
# Convert each sentence into a sequence of integers
sequence = tokenizer.texts_to_sequences(texts)
# Find the length of the longest sequence
max_length = max([len(seq) for seq in sequence])
# Pad all sequences to the same length
padded_sequence = pad_sequences(sequence,maxlen=max_length)

In [23]:
#Encode the string labels into integers
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(lables)
print(labels[:5])

[4 0 3 0 4]


In [24]:
# One-hot enode the labels
# Convert integer class labels into one-hot encoded vectors

one_hot_labels = keras.utils.to_categorical(labels)
print(one_hot_labels[:5])

[[0. 0. 0. 0. 1. 0.]
 [1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0.]]


In [25]:
xtrain,xtest,ytrain,ytest = train_test_split(padded_sequence, one_hot_labels, test_size=0.2, random_state=42)

In [26]:
#Define the modelm

model = Sequential()
model.add(Embedding(input_dim=len(tokenizer.word_index)+1,
                    output_dim=128, input_length = max_length))
model.add(Flatten())
# model.add(Dropout(0.5))
model.add(Dense(units=128, activation="relu"))
model.add(Dense(units=len(one_hot_labels[0]),activation="softmax"))

model.compile(optimizer = "adam", loss="categorical_crossentropy",metrics = ["accuracy"])

model.fit(xtrain,ytrain, epochs = 10, batch_size=32, validation_data=(xtest,ytest))

Epoch 1/10


c:\A_Computer_Main\main\all_work\vs-studio-saves\datascience_project\project_env\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


400/400 ━━━━━━━━━━━━━━━━━━━━ 21s 46ms/step - accuracy: 0.4951 - loss: 1.3307 - val_accuracy: 0.7159 - val_loss: 0.8123
Epoch 2/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 19s 46ms/step - accuracy: 0.9021 - loss: 0.3163 - val_accuracy: 0.8119 - val_loss: 0.5359
Epoch 3/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 21s 46ms/step - accuracy: 0.9850 - loss: 0.0579 - val_accuracy: 0.8213 - val_loss: 0.5600
Epoch 4/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 17s 43ms/step - accuracy: 0.9943 - loss: 0.0276 - val_accuracy: 0.8175 - val_loss: 0.6037
Epoch 5/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 19s 47ms/step - accuracy: 0.9965 - loss: 0.0191 - val_accuracy: 0.8244 - val_loss: 0.6204
Epoch 6/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 18s 46ms/step - accuracy: 0.9961 - loss: 0.0179 - val_accuracy: 0.8197 - val_loss: 0.6987
Epoch 7/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 14s 36ms/step - accuracy: 0.9968 - loss: 0.0151 - val_accuracy: 0.8172 - val_loss: 0.7133
Epoch 8/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 14s 34ms/step - accuracy: 0.9971 - loss: 0.0129 - val_accurac

In [30]:

# Preprocess the input texts
def preprocess_input_text(input_text):
    input_sequence = tokenizer.texts_to_sequences([input_text])
    padded_input_sequence = pad_sequences(input_sequence, maxlen=max_length)
    prediction = model.predict(padded_input_sequence)

    predicted_label = label_encoder.inverse_transform([np.argmax(prediction[0])])
    print(predicted_label[0])

In [31]:
input_text = "she didn't come today because she lost her dog yesterday"
preprocess_input_text(input_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
sadness


In [32]:
input_text = "i used to be able to hang around talk with the cashier when i was putting away my money now i feel rushed and stressed if i take a second to fumble with the coins and put them in my purse"
preprocess_input_text(input_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
anger
